## tl;dr
과거 선취매 전체 모델의 검증은 자료 부족으로 미완료입니다. 기업행위 원천 파일이 없고 거래상태 6일·점수 기준일 14일만 보유합니다. 코드 가드 및 표본 불확실성 보완과 원천 검증 완료를 구분합니다.
## Context & Methods
이 노트북은 로컬 immutable 감사 결과를 읽는 재현 동반자료입니다. 원천 수집·점수 변경·공개 배포는 하지 않습니다.
### Key Assumptions
연도별 상승 여부의 Wilson 구간은 독립 베르누이 가정의 참고치이지 매매 성공확률이 아닙니다. 날짜 열의 존재는 당시 이용 가능성의 증명이 아닙니다. 방법: https://www.itl.nist.gov/div898/handbook/prc/section2/prc241.htm


## Data
아래 ROOT를 프로젝트 위치에 맞춰 수정하세요. `.venv` Python의 프로젝트 의존성이 필요합니다.

In [ ]:
from pathlib import Path
import sys
ROOT = Path('C:/Users/a4jud/kr_quant_research')
sys.path.insert(0, str(ROOT/'src'))
from kr_quant.research.selection_ledger import read_verified
audit = read_verified(ROOT/'output/audit/pit-step3-20260908.json')['payload']
assert audit['verified'] is False
for name, source in audit['sources'].items():
    print(name, source['status'], source.get('rows'), {k:v['distinct_days'] for k,v in source.get('date_ranges',{}).items()})


## Results
3개년 모두 상승한 경우의 참고 구간과 다중 탐색 보정. 보정 p값이 작더라도 전체 모델 검증이 되지는 않습니다.

In [ ]:
from kr_quant.research.statistical_reliability import sample_reliability
result = sample_reliability([{'year':y, 'return':0.1} for y in [2021,2022,2023]], family_size=1200)
assert abs(result['wilson95'][0] - 0.438502968) < 1e-8
assert result['p_bonferroni_reference'] == 1
print({k:result[k] for k in ['n','wins','wilson95','p_raw_reference','p_bonferroni_reference','verified']})


## Takeaways
운영 최신화와 시간상 분리 검사는 가능하지만, 당시 상장폐지 포함 종목 집합·기업행위·정정 공시 버전의 원래 이용 가능성이 확보돼야 전체 모델 OOS 검증으로 나아갈 수 있습니다. 현재 참고 진단을 검증 성과로 승격시키지 않습니다.
